In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive mounted")

In [ ]:
import sys
import os

# Add project directory to Python path
PROJECT_DIR = "/content/drive/MyDrive/secureflow_analytics"
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Create project directory if it doesn't exist
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f"Project directory: {PROJECT_DIR}")

# SecureFlow Analytics - Synthetic Data Generation

This notebook generates realistic product telemetry data for a security software application.

## Key Patterns Encoded for Interview Insights

| Pattern | Description | Analysis Method |
|---------|-------------|----------------|
| Onboarding → Retention | Users with <3 scans in week 1 churn 2-3x more | Cohort Analysis |
| Feature → Churn | real_time_protection adopters have 40% lower churn | Feature Importance |
| A/B Test (Positive) | New onboarding flow: +15% activation | t-test, CUPED |
| A/B Test (Negative) | Early upsell: -25% conversion | Learning from failures |
| Causal Inference | Geo-based rollout confounded by country | Difference-in-Differences |
| Channel → LTV | Referral users have 25% higher LTV | CAC/LTV analysis |

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from pathlib import Path
import json
import warnings

warnings.filterwarnings('ignore')

# Change to project directory
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())

In [ ]:
# Import all configuration from config.py
from config import (
    # Paths
    RAW_DIR,
    
    # Data generation params
    RANDOM_SEED,
    DATE_START,
    DATE_END,
    N_USERS,
    
    # Categorical values
    PLAN_TYPES,
    ACQUISITION_CHANNELS,
    COUNTRIES,
    DEVICE_OS,
    EVENT_TYPES,
    FEATURES,
    THREAT_TYPES,
    THREAT_SEVERITIES,
    SCAN_TYPES,
    CANCELLATION_REASONS,
    TICKET_CATEGORIES,
    
    # Channel params
    CHANNEL_CONVERSION_MODIFIER,
    CHANNEL_CHURN_MODIFIER,
    
    # Plan params
    PLAN_CHURN_RATES,
    PLAN_MRR,
    
    # Experiment configs
    EXPERIMENTS,
    
    # Feature params
    FEATURE_ADOPTION_RATES,
    FEATURE_CHURN_IMPACT,
    PREMIUM_ONLY_FEATURES,
    
    # Event probabilities
    EVENT_PROBS_ONBOARDING,
    EVENT_PROBS_THREAT,
    EVENT_PROBS_NORMAL,
    
    # Usage patterns
    HOUR_WEIGHTS_DESKTOP,
    HOUR_WEIGHTS_MOBILE,
    APP_VERSIONS,
)

# Set random seed
np.random.seed(RANDOM_SEED)

# Create output directory
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Parse dates
START_DATE = datetime.strptime(DATE_START, "%Y-%m-%d")
END_DATE = datetime.strptime(DATE_END, "%Y-%m-%d")

print(f"Output directory: {RAW_DIR}")
print(f"Date range: {DATE_START} to {DATE_END}")
print(f"User count: {N_USERS:,}")
print("\n✓ Configuration loaded successfully!")

## 2. Generate Users

In [ ]:
def generate_users(n_users: int) -> pd.DataFrame:
    """
    Generate user profiles with intentional patterns.
    
    KEY RELATIONSHIPS:
    - acquisition_channel affects conversion and LTV
    - device_os affects engagement patterns
    - country affects feature availability (for DiD)
    - Q4 signups have higher conversion (promotions)
    """
    
    # Email domains for realism
    email_domains = ["gmail.com", "yahoo.com", "outlook.com", "hotmail.com", "icloud.com"]
    
    users = []
    days_range = (END_DATE - START_DATE).days
    
    for i in range(n_users):
        user_id = f"user_{i:06d}"
        
        # Signup date - leave 30 days for events after signup
        signup_date = START_DATE + timedelta(
            days=np.random.randint(0, days_range - 30)
        )
        
        # Channel selection
        channel = np.random.choice(
            list(ACQUISITION_CHANNELS.keys()),
            p=list(ACQUISITION_CHANNELS.values())
        )
        
        country = np.random.choice(
            list(COUNTRIES.keys()),
            p=list(COUNTRIES.values())
        )
        
        device_os = np.random.choice(
            list(DEVICE_OS.keys()),
            p=list(DEVICE_OS.values())
        )
        
        # Plan type - affected by channel
        base_premium_prob = 0.15
        premium_prob = base_premium_prob * CHANNEL_CONVERSION_MODIFIER.get(channel, 1.0)
        
        # Q4 signups more likely to be premium (promotions)
        if signup_date.month in [10, 11, 12]:
            premium_prob *= 1.3
            
        premium_prob = min(premium_prob, 0.5)  # Cap at 50%
        
        plan_type = np.random.choice(
            ['free', 'premium_monthly', 'premium_annual'],
            p=[1 - premium_prob, premium_prob * 0.6, premium_prob * 0.4]
        )
        
        users.append({
            'user_id': user_id,
            'signup_date': signup_date.strftime('%Y-%m-%d'),
            'plan_type': plan_type,
            'country': country,
            'device_os': device_os,
            'acquisition_channel': channel,
            'email_domain': np.random.choice(email_domains),
        })
    
    return pd.DataFrame(users)


# Generate users
print("Generating users...")
users_df = generate_users(N_USERS)
print(f"✓ Generated {len(users_df):,} users")
print(f"\nPlan distribution:")
print(users_df['plan_type'].value_counts())
print(f"\nChannel distribution:")
print(users_df['acquisition_channel'].value_counts())
users_df.head()

## 3. Generate Subscriptions

In [ ]:
def generate_subscriptions(users_df: pd.DataFrame) -> pd.DataFrame:
    """
    Generate subscription history with churn patterns.
    
    KEY PATTERNS:
    - Referral users have lower churn
    - Q4 subscribers have higher retention
    - Annual plans have much lower churn
    """
    
    subscriptions = []
    
    for _, user in users_df.iterrows():
        user_id = user['user_id']
        signup_date = datetime.strptime(user['signup_date'], '%Y-%m-%d')
        plan_type = user['plan_type']
        channel = user['acquisition_channel']
        
        # Base churn from plan
        base_churn = PLAN_CHURN_RATES.get(plan_type, 0.10)
        
        # Channel modifier
        churn_modifier = CHANNEL_CHURN_MODIFIER.get(channel, 1.0)
        
        # Q4 cohorts have better retention
        if signup_date.month in [10, 11, 12]:
            churn_modifier *= 0.85
        
        final_churn = base_churn * churn_modifier
        
        # MRR
        mrr = PLAN_MRR.get(plan_type, 0)
        
        # Simulate subscription lifecycle
        current_date = signup_date
        sub_id = 0
        
        while current_date < END_DATE:
            sub_id += 1
            start_date = current_date
            
            # Period length
            if plan_type == 'premium_annual':
                period_days = 365
            else:
                period_days = 30
            
            # Random churn within period
            if np.random.random() < final_churn:
                churn_day = np.random.randint(7, period_days)
                end_date = start_date + timedelta(days=churn_day)
                status = 'cancelled'
            else:
                end_date = start_date + timedelta(days=period_days)
                status = 'active' if end_date >= END_DATE else 'renewed'
            
            # Cancellation reason
            cancel_reason = None
            if status == 'cancelled':
                cancel_reason = np.random.choice(
                    ['price', 'not_using', 'switched_competitor', 'technical_issues'],
                    p=[0.30, 0.40, 0.18, 0.12]
                )
            
            subscriptions.append({
                'subscription_id': f"{user_id}_sub_{sub_id:02d}",
                'user_id': user_id,
                'plan': plan_type,
                'start_date': start_date.strftime('%Y-%m-%d'),
                'end_date': min(end_date, END_DATE).strftime('%Y-%m-%d'),
                'status': status,
                'mrr': mrr,
                'cancellation_reason': cancel_reason
            })
            
            if status == 'cancelled':
                break
            current_date = end_date
    
    return pd.DataFrame(subscriptions)


# Generate subscriptions
print("Generating subscriptions...")
subscriptions_df = generate_subscriptions(users_df)
print(f"✓ Generated {len(subscriptions_df):,} subscription records")
print(f"\nStatus distribution:")
print(subscriptions_df.groupby('user_id')['status'].last().value_counts())
subscriptions_df.head()

## 4. Generate Events

In [ ]:
def get_hour_weights(device: str) -> list:
    """Get hour weights based on device type."""
    if device in ['windows', 'macos']:
        weights = HOUR_WEIGHTS_DESKTOP
    else:
        weights = HOUR_WEIGHTS_MOBILE
    # Normalize
    total = sum(weights)
    return [w/total for w in weights]


def get_event_probs(phase: str, plan: str, engagement: str) -> list:
    """Get event probabilities based on user state."""
    if phase == 'onboarding':
        probs = EVENT_PROBS_ONBOARDING
    elif phase == 'threat':
        probs = EVENT_PROBS_THREAT
    else:
        probs = EVENT_PROBS_NORMAL.copy()
        
        # Free users see more upgrade prompts
        if plan == 'free':
            probs['upgrade_clicked'] = 0.08
        
        # High engagement = more feature discovery
        if engagement == 'high':
            probs['feature_discovered'] = 0.10
    
    # Convert to list and normalize
    prob_list = [probs.get(et, 0.05) for et in EVENT_TYPES]
    total = sum(prob_list)
    return [p/total for p in prob_list]


def get_app_version(date: datetime) -> str:
    """Get app version based on date."""
    for start, end, version in APP_VERSIONS:
        start_dt = datetime.strptime(start, '%Y-%m-%d')
        end_dt = datetime.strptime(end, '%Y-%m-%d')
        if start_dt <= date <= end_dt:
            return version
    return '4.2.0'


def generate_event_properties(event_type: str, day_num: int) -> dict:
    """Generate contextual event properties."""
    props = {}
    
    if event_type == 'scan_started':
        props['scan_type'] = np.random.choice(SCAN_TYPES, p=[0.6, 0.3, 0.1])
        
    elif event_type == 'scan_completed':
        props['scan_type'] = np.random.choice(SCAN_TYPES, p=[0.6, 0.3, 0.1])
        props['duration_seconds'] = int(np.random.exponential(120))
        props['files_scanned'] = int(np.random.exponential(5000))
        props['threats_found'] = int(np.random.poisson(0.3))
        
    elif event_type == 'threat_detected':
        props['threat_type'] = np.random.choice(
            THREAT_TYPES, p=[0.15, 0.25, 0.20, 0.15, 0.25]
        )
        props['severity'] = np.random.choice(
            THREAT_SEVERITIES, p=[0.3, 0.4, 0.2, 0.1]
        )
        
    elif event_type == 'threat_resolved':
        props['action'] = np.random.choice(
            ['quarantined', 'deleted', 'allowed'], p=[0.5, 0.4, 0.1]
        )
        
    elif event_type == 'settings_changed':
        props['setting'] = np.random.choice([
            'real_time_protection', 'scan_schedule', 'notifications',
            'firewall_rules', 'exclusions'
        ])
        
    elif event_type == 'upgrade_clicked':
        props['source'] = np.random.choice(
            ['banner', 'feature_gate', 'settings', 'notification']
        )
        props['plan_shown'] = np.random.choice(['monthly', 'annual'])
        
    elif event_type == 'help_viewed':
        props['article_id'] = f"help_{np.random.randint(1, 50):03d}"
        props['category'] = np.random.choice(
            ['setup', 'scanning', 'threats', 'billing', 'features']
        )
        
    elif event_type == 'onboarding_step_completed':
        step = min(day_num, 5)
        props['step'] = step
        props['step_name'] = [
            'welcome', 'first_scan', 'review_results',
            'configure_settings', 'complete'
        ][step-1] if step > 0 else 'welcome'
    
    return props

In [ ]:
def generate_events(users_df: pd.DataFrame, subscriptions_df: pd.DataFrame) -> pd.DataFrame:
    """
    Generate product telemetry events with behavioral patterns.
    
    KEY PATTERNS:
    - Users who complete onboarding (3+ scans in week 1) retain better
    - threat_detected without threat_resolved correlates with churn
    - help_viewed frequency indicates confusion/friction
    """
    
    # Get user active periods
    user_periods = subscriptions_df.groupby('user_id').agg({
        'start_date': 'min',
        'end_date': 'max',
        'status': 'last'
    }).reset_index()
    
    users_merged = users_df.merge(user_periods, on='user_id')
    
    all_events = []
    event_counter = 0
    
    # Progress tracking
    total_users = len(users_merged)
    
    for idx, user in users_merged.iterrows():
        if idx % 5000 == 0:
            print(f"  Processing user {idx:,}/{total_users:,}...")
        
        user_id = user['user_id']
        start = datetime.strptime(user['start_date'], '%Y-%m-%d')
        end = datetime.strptime(user['end_date'], '%Y-%m-%d')
        is_churned = user['status'] == 'cancelled'
        plan = user['plan_type']
        device = user['device_os']
        
        # User engagement level
        if is_churned:
            engagement = np.random.choice(['low', 'medium'], p=[0.7, 0.3])
        else:
            engagement = np.random.choice(['low', 'medium', 'high'], p=[0.2, 0.5, 0.3])
        
        # Events per day based on engagement
        events_per_day = {'low': 0.5, 'medium': 2, 'high': 5}[engagement]
        
        # Generate events
        current = start
        day_num = 0
        onboarding_complete = False
        scans_week1 = 0
        unresolved_threats = 0
        
        while current <= end:
            day_num += 1
            n_events = np.random.poisson(events_per_day)
            
            for _ in range(n_events):
                hour = np.random.choice(range(24), p=get_hour_weights(device))
                timestamp = current + timedelta(
                    hours=hour, 
                    minutes=np.random.randint(0, 60)
                )
                
                # Determine phase for event probabilities
                if day_num <= 3 and not onboarding_complete:
                    phase = 'onboarding'
                elif unresolved_threats > 0:
                    phase = 'threat'
                else:
                    phase = 'normal'
                
                event_probs = get_event_probs(phase, plan, engagement)
                event_type = np.random.choice(EVENT_TYPES, p=event_probs)
                
                # Track states
                if event_type == 'scan_completed' and day_num <= 7:
                    scans_week1 += 1
                    if scans_week1 >= 3:
                        onboarding_complete = True
                
                if event_type == 'threat_detected':
                    unresolved_threats += 1
                elif event_type == 'threat_resolved':
                    unresolved_threats = max(0, unresolved_threats - 1)
                
                # Generate properties
                properties = generate_event_properties(event_type, day_num)
                
                all_events.append({
                    'event_id': f"evt_{event_counter:010d}",
                    'user_id': user_id,
                    'timestamp': timestamp.strftime('%Y-%m-%d %H:%M:%S'),
                    'event_type': event_type,
                    'properties': json.dumps(properties),
                    'session_id': f"sess_{user_id}_{current.strftime('%Y%m%d')}_{np.random.randint(1,5):02d}",
                    'device_os': device,
                    'app_version': get_app_version(current)
                })
                event_counter += 1
            
            current += timedelta(days=1)
    
    return pd.DataFrame(all_events)


# Generate events
print("Generating events (this may take a few minutes)...")
events_df = generate_events(users_df, subscriptions_df)
print(f"\n✓ Generated {len(events_df):,} events")
print(f"\nEvent type distribution:")
print(events_df['event_type'].value_counts())

## 5. Generate Experiments

In [ ]:
def generate_experiments(users_df: pd.DataFrame) -> tuple:
    """
    Generate A/B test assignments and metrics.
    
    EXPERIMENTS WITH INTENTIONAL OUTCOMES:
    - exp_001: +15% activation (positive)
    - exp_002: -20% help views (positive - less confusion)
    - exp_003: -25% conversion (negative - earlier upsell hurts)
    - exp_004: Confounded by country (needs DiD)
    """
    
    experiments_list = []
    assignments = []
    metrics = []
    
    for exp_id, exp_config in EXPERIMENTS.items():
        experiments_list.append({
            'experiment_id': exp_id,
            'experiment_name': exp_config['name'],
            'start_date': exp_config['start_date'],
            'end_date': exp_config['end_date'],
            'status': 'completed',
            'variants': str(exp_config['variants']),
            'allocation': str(exp_config['allocation']),
            'primary_metric': exp_config['primary_metric'],
            'targeting': exp_config['targeting']
        })
        
        # Get eligible users
        start = datetime.strptime(exp_config['start_date'], '%Y-%m-%d')
        end = datetime.strptime(exp_config['end_date'], '%Y-%m-%d')
        
        eligible = users_df[
            (pd.to_datetime(users_df['signup_date']) >= start) &
            (pd.to_datetime(users_df['signup_date']) <= end)
        ].copy()
        
        # Filter by targeting
        if exp_config['targeting'] == 'free_users':
            eligible = eligible[eligible['plan_type'] == 'free']
        
        # Assign variants
        for _, user in eligible.iterrows():
            # exp_004 is non-randomized (by country)
            if exp_id == 'exp_004':
                variant = 'rollout_group' if user['country'] in ['US', 'CA'] else 'control_group'
            else:
                variant = np.random.choice(
                    exp_config['variants'],
                    p=exp_config['allocation']
                )
            
            assignments.append({
                'experiment_id': exp_id,
                'user_id': user['user_id'],
                'variant': variant,
                'assigned_at': user['signup_date']
            })
            
            # Generate metric outcome
            if exp_id == 'exp_001':  # Onboarding: +15% activation
                base_rate = 0.45
                if variant == 'treatment':
                    rate = base_rate * (1 + exp_config['expected_lift'])
                else:
                    rate = base_rate
                metric_value = 1 if np.random.random() < rate else 0
                metric_name = 'activated'
                
            elif exp_id == 'exp_002':  # Threat explainer: -20% help views
                base_rate = 0.35
                if variant == 'treatment':
                    rate = base_rate * (1 + exp_config['expected_lift'])  # -0.20 = reduction
                else:
                    rate = base_rate
                metric_value = 1 if np.random.random() < rate else 0
                metric_name = 'viewed_help'
                
            elif exp_id == 'exp_003':  # Early upsell: -25% conversion (negative!)
                base_rate = 0.12
                if variant == 'day_3_upsell':
                    rate = base_rate * (1 + exp_config['expected_lift'])  # -0.25 = worse
                else:
                    rate = base_rate
                metric_value = 1 if np.random.random() < rate else 0
                metric_name = 'converted_to_premium'
                
            elif exp_id == 'exp_004':  # Real-time default: confounded
                base_rate = 0.70
                # Country effect (confound)
                country_effect = 0.05 if user['country'] in ['US', 'CA'] else 0
                # Treatment effect
                treatment_effect = 0.10 if variant == 'rollout_group' else 0
                rate = base_rate + country_effect + treatment_effect
                metric_value = 1 if np.random.random() < rate else 0
                metric_name = 'threat_resolved'
            
            metrics.append({
                'experiment_id': exp_id,
                'user_id': user['user_id'],
                'metric_name': metric_name,
                'metric_value': metric_value,
                'recorded_at': user['signup_date']
            })
    
    return (
        pd.DataFrame(experiments_list),
        pd.DataFrame(assignments),
        pd.DataFrame(metrics)
    )


# Generate experiments
print("Generating experiments...")
experiments_df, exp_assignments_df, exp_metrics_df = generate_experiments(users_df)
print(f"✓ Generated {len(experiments_df)} experiments")
print(f"✓ Generated {len(exp_assignments_df):,} assignments")
print(f"✓ Generated {len(exp_metrics_df):,} metric records")
experiments_df

## 6. Generate Feature Usage

In [ ]:
def generate_feature_usage(users_df: pd.DataFrame, subscriptions_df: pd.DataFrame) -> pd.DataFrame:
    """
    Generate feature usage data with adoption patterns.
    
    KEY PATTERNS:
    - real_time_protection users have 40% lower churn
    - vpn_feature has low adoption but high value
    - password_manager adoption varies by channel
    """
    
    # Get churn status
    user_status = subscriptions_df.groupby('user_id')['status'].last().reset_index()
    users_merged = users_df.merge(user_status, on='user_id')
    
    usage_records = []
    
    for _, user in users_merged.iterrows():
        user_id = user['user_id']
        plan = user['plan_type']
        channel = user['acquisition_channel']
        is_churned = user['status'] == 'cancelled'
        signup = datetime.strptime(user['signup_date'], '%Y-%m-%d')
        
        for feature in FEATURES:
            # Skip premium features for free users
            if feature in PREMIUM_ONLY_FEATURES and plan == 'free':
                continue
            
            # Base adoption rate
            base_adoption = FEATURE_ADOPTION_RATES.get(feature, 0.30)
            
            # Adjust based on churn (reverse causality: adopters don't churn)
            if feature == 'real_time_protection':
                # Non-churned users more likely to have adopted
                if not is_churned:
                    base_adoption = 0.80
                else:
                    base_adoption = 0.50
            
            # Channel effect for password_manager
            if feature == 'password_manager':
                if channel == 'referral':
                    base_adoption = 0.35
                elif channel == 'organic_search':
                    base_adoption = 0.25
                else:
                    base_adoption = 0.20
            
            # Determine if user adopted
            if np.random.random() < base_adoption:
                first_use = signup + timedelta(days=np.random.randint(0, 30))
                
                # Usage intensity
                if is_churned:
                    usage_count = int(np.random.exponential(5))
                else:
                    usage_count = int(np.random.exponential(20))
                
                last_used = first_use + timedelta(days=np.random.randint(0, 180))
                
                usage_records.append({
                    'user_id': user_id,
                    'feature_name': feature,
                    'first_used_at': first_use.strftime('%Y-%m-%d'),
                    'usage_count': max(1, usage_count),
                    'last_used_at': min(last_used, END_DATE).strftime('%Y-%m-%d')
                })
    
    return pd.DataFrame(usage_records)


# Generate feature usage
print("Generating feature usage...")
feature_usage_df = generate_feature_usage(users_df, subscriptions_df)
print(f"✓ Generated {len(feature_usage_df):,} feature usage records")
print(f"\nFeature adoption:")
print(feature_usage_df['feature_name'].value_counts())

## 7. Generate Support Tickets

In [ ]:
def generate_support_tickets(users_df: pd.DataFrame, events_df: pd.DataFrame) -> pd.DataFrame:
    """
    Generate support tickets correlated with user experience.
    
    PATTERN: Users with high help_viewed events are more likely to submit tickets.
    """
    
    # Count help_viewed events per user
    help_counts = events_df[events_df['event_type'] == 'help_viewed'].groupby('user_id').size()
    
    tickets = []
    
    for _, user in users_df.iterrows():
        user_id = user['user_id']
        signup = datetime.strptime(user['signup_date'], '%Y-%m-%d')
        
        # Ticket probability based on help views
        help_views = help_counts.get(user_id, 0)
        ticket_prob = min(0.5, 0.05 + help_views * 0.02)
        
        if np.random.random() < ticket_prob:
            n_tickets = np.random.poisson(1) + 1
            
            for i in range(n_tickets):
                ticket_date = signup + timedelta(days=np.random.randint(1, 180))
                
                if ticket_date > END_DATE:
                    continue
                
                tickets.append({
                    'ticket_id': f"ticket_{len(tickets):06d}",
                    'user_id': user_id,
                    'created_at': ticket_date.strftime('%Y-%m-%d'),
                    'category': np.random.choice(
                        TICKET_CATEGORIES,
                        p=[0.35, 0.20, 0.15, 0.20, 0.10]
                    ),
                    'priority': np.random.choice(
                        ['low', 'medium', 'high'],
                        p=[0.4, 0.45, 0.15]
                    ),
                    'resolution_hours': int(np.random.exponential(24)),
                    'csat_score': np.random.choice(
                        [1, 2, 3, 4, 5],
                        p=[0.05, 0.10, 0.20, 0.35, 0.30]
                    )
                })
    
    return pd.DataFrame(tickets)


# Generate support tickets
print("Generating support tickets...")
tickets_df = generate_support_tickets(users_df, events_df)
print(f"✓ Generated {len(tickets_df):,} support tickets")
print(f"\nCategory distribution:")
print(tickets_df['category'].value_counts())

## 8. Validation

In [ ]:
def validate_data():
    """Run validation checks."""
    print("=" * 60)
    print("DATA VALIDATION REPORT")
    print("=" * 60)
    
    print(f"\n📊 ROW COUNTS:")
    print(f"  Users:                  {len(users_df):,}")
    print(f"  Subscriptions:          {len(subscriptions_df):,}")
    print(f"  Events:                 {len(events_df):,}")
    print(f"  Experiments:            {len(experiments_df)}")
    print(f"  Experiment Assignments: {len(exp_assignments_df):,}")
    print(f"  Experiment Metrics:     {len(exp_metrics_df):,}")
    print(f"  Feature Usage:          {len(feature_usage_df):,}")
    print(f"  Support Tickets:        {len(tickets_df):,}")
    
    print(f"\n📈 CHURN RATE:")
    churn_status = subscriptions_df.groupby('user_id')['status'].last()
    churn_rate = (churn_status == 'cancelled').mean() * 100
    print(f"  Overall: {churn_rate:.1f}%")
    
    print(f"\n🔬 EXPERIMENT METRICS (expected vs actual):")
    for exp_id, config in EXPERIMENTS.items():
        exp_data = exp_metrics_df[exp_metrics_df['experiment_id'] == exp_id]
        assignments = exp_assignments_df[exp_assignments_df['experiment_id'] == exp_id]
        merged = exp_data.merge(assignments[['user_id', 'variant']], on='user_id')
        
        if len(merged) > 0:
            rates = merged.groupby('variant')['metric_value'].mean()
            print(f"  {exp_id} ({config['name']}):")
            for variant, rate in rates.items():
                print(f"    {variant}: {rate:.3f}")
    
    print(f"\n✅ REFERENTIAL INTEGRITY:")
    events_users = set(events_df['user_id'].unique())
    all_users = set(users_df['user_id'].unique())
    orphan_events = events_users - all_users
    print(f"  Orphan events: {len(orphan_events)} (should be 0)")
    
    print("\n" + "=" * 60)


validate_data()

## 9. Export to Parquet

In [ ]:
# Save to parquet (more efficient than CSV for large files)
print("Saving data to parquet...")

users_df.to_parquet(RAW_DIR / 'users.parquet', index=False)
subscriptions_df.to_parquet(RAW_DIR / 'subscriptions.parquet', index=False)
events_df.to_parquet(RAW_DIR / 'events.parquet', index=False)
experiments_df.to_parquet(RAW_DIR / 'experiments.parquet', index=False)
exp_assignments_df.to_parquet(RAW_DIR / 'experiment_assignments.parquet', index=False)
exp_metrics_df.to_parquet(RAW_DIR / 'experiment_metrics.parquet', index=False)
feature_usage_df.to_parquet(RAW_DIR / 'feature_usage.parquet', index=False)
tickets_df.to_parquet(RAW_DIR / 'support_tickets.parquet', index=False)

print(f"\n✅ Data exported successfully!")
print(f"\nFiles saved to: {RAW_DIR}")

# Show file sizes
for f in RAW_DIR.glob("*.parquet"):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name}: {size_mb:.2f} MB")

In [ ]:
# Show directory structure
import os

print(f"\n📁 Project Structure:")
for root, dirs, files in os.walk(str(RAW_DIR.parent)):
    level = root.replace(str(RAW_DIR.parent), "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")

## 10. Summary

### Embedded Patterns for Interview Discussion

| Analysis | Pattern | Expected Finding |
|----------|---------|------------------|
| **Churn Prediction** | Onboarding completion | <3 scans in week 1 → 2-3x higher churn |
| **Feature Impact** | real_time_protection | Adopters have ~40% lower churn |
| **A/B Test (exp_001)** | New onboarding flow | +15% activation (ship it!) |
| **A/B Test (exp_002)** | Threat explainer | -20% help views (less confusion) |
| **A/B Test (exp_003)** | Early upsell | -25% conversion (don't ship!) |
| **Causal Inference (exp_004)** | Geo rollout | Confounded - needs DiD |
| **Cohort Analysis** | Q4 signups | Higher retention (promotions) |
| **Channel Analysis** | Referral users | 25% higher LTV |

### Next Steps

1. **Phase 1**: EDA & Data Validation
2. **Phase 2**: Funnel & Cohort Analysis
3. **Phase 3**: Experimentation Framework
4. **Phase 4**: Causal Inference (DiD for exp_004)
5. **Phase 5**: Churn Prediction Model
6. **Phase 6**: MLOps & Deployment
7. **Phase 7**: Dashboards